In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from unidecode import unidecode
import forestplot
import matplotlib.pyplot as plt

In [209]:
# load data
rais = pd.read_csv("final_project_data/rais_rmsp.csv")

ibge_raw = pd.read_csv("final_project_data/Agregados_por_setores_basico_BR.csv", encoding="latin1", sep=";", quotechar='"', decimal=",")
rmsp_codes = rais["id_municipio"].tolist()
ibge = ibge_raw[ibge_raw["CD_MUN"].isin(rmsp_codes)]

od = pd.read_spss("final_project_data/Banco2023_divulgacao_190225.sav")

In [ ]:
zone_crosswalk = pd.read_excel("final_project_data/CORRESPONDׂNCIA ENTRE ZONAS 2007 e 2017.xlsx",sheet_name="Divisão Administrativa",header=5,usecols="A:G")

In [259]:
sp_city = rais[rais["id_municipio"] == "3550308"]
rmsp_other = rais[rais["id_municipio"] != "3550308"]

jobs_by_district = sp_city.groupby("distrito_clean")["quantidade_vinculos_ativos"].sum().reset_index()
jobs_by_muni = rmsp_other.groupby("id_municipio")["quantidade_vinculos_ativos"].sum().reset_index()
rmsp_other

,ano,sigla_uf,id_municipio,quantidade_vinculos_ativos,quantidade_vinculos_clt,quantidade_vinculos_estatutarios,natureza_estabelecimento,natureza_juridica,tamanho_estabelecimento,tipo_estabelecimento,...,cnae_2_subclasse,subsetor_ibge,subatividade_ibge,cep,bairros_sp,distritos_sp,bairros_fortaleza,bairros_rj,regioes_administrativas_df,distrito_clean
0,2021,SP,3550308,14,14,0,NaN,4120,4,3,...,121101,25,NaN,4895020,200.0,55.0,NaN,NaN,NaN,55.0
1,2021,SP,3552502,73,73,0,NaN,4022,6,2,...,121101,25,NaN,8620080,9999.0,9999.0,NaN,NaN,NaN,9999.0
2,2021,SP,3530607,17,17,0,NaN,4120,4,2,...,121101,25,NaN,8710970,9999.0,9999.0,NaN,NaN,NaN,9999.0
3,2021,SP,3518305,13,13,0,NaN,4081,4,2,...,122900,25,NaN,8900000,9999.0,9999.0,NaN,NaN,NaN,9999.0
4,2021,SP,3530607,12,12,0,NaN,4120,4,2,...,122900,25,NaN,8767300,9999.0,9999.0,NaN,NaN,NaN,9999.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1272729,2021,SP,3550308,11,11,0,NaN,2062,4,1,...,9602502,21,NaN,4534001,240.0,33.0,NaN,NaN,NaN,33.0
1272730,2021,SP,3550308,11,11,0,NaN,2305,4,1,...,9602502,21,NaN,4090000,246.0,51.0,NaN,NaN,NaN,51.0
1272731,2021,SP,3550308,11,11,0,NaN,2062,4,1,...,9603301,21,NaN,4548005,1716.0,33.0,NaN,NaN,NaN,33.0
1272732,2021,SP,3550308,11,11,0,NaN,2305,4,1,...,9609208,21,NaN,5339004,1605.0,67.0,NaN,NaN,NaN,67.0


In [260]:
sp_zones = zone_crosswalk.merge(jobs_by_district, left_on="distrito_num", right_on="distrito_clean", how="left")
sp_zones["zones_in_distrito"] = sp_zones.groupby("distrito_num")["zona_num"].transform("count")
sp_zones["jobs_allocated"] = sp_zones["quantidade_vinculos_ativos"] / sp_zones["zones_in_distrito"]

# other_zones = zone_crosswalk.merge(jobs_by_muni, left_on="muni_num", right_on="id_municipio", how="left")
# other_zones["zones_in_muni"] = other_zones.groupby("muni_num")["zona_num"].transform("count")
# other_zones["jobs_allocated"] = other_zones["quantidade_vinculos_ativos"] / other_zones["zones_in_muni"]

other

# jobs_by_zone = pd.concat([
#     sp_zones[["zona_num", "jobs_allocated"]],
#     other_zones[["zona_num", "jobs_allocated"]]
# ]).dropna(subset=["zona_num"])

,zona_num,zona_nome,muni_num,muni_nome,distrito_num,distrito_nome,area_ha,dist_name_clean,distrito_clean,quantidade_vinculos_ativos,zones_in_distrito,jobs_allocated
0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NAN,NaN,NaN,0,NaN
1,1.0,Sé,36.0,São Paulo,80.0,Sé,57.10,SE,NaN,NaN,3,NaN
2,2.0,Parque Dom Pedro,36.0,São Paulo,80.0,Sé,113.64,SE,NaN,NaN,3,NaN
3,3.0,Praça João Mendes,36.0,São Paulo,80.0,Sé,47.75,SE,NaN,NaN,3,NaN
4,4.0,Ladeira da Memória,36.0,São Paulo,67.0,República,75.11,REPUBLICA,NaN,NaN,3,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
525,525.0,Amador Bueno,17.0,Itapevi,113.0,Itapevi,5113.23,ITAPEVI,NaN,NaN,3,NaN
526,526.0,Santana de Parnaíba,31.0,Santana de Parnaíba,127.0,Santana de Parnaíba,18034.76,SANTANA DE PARNAIBA,NaN,NaN,1,NaN
527,527.0,Pirapora do Bom Jesus,25.0,Pirapora do Bom Jesus,121.0,Pirapora do Bom Jesus,10876.89,PIRAPORA DO BOM JESUS,NaN,NaN,1,NaN
528,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NAN,NaN,NaN,0,NaN


In [254]:
rmsp_codes_str = [str(c) for c in rmsp_codes]
ibge_rmsp = ibge[ibge["CD_MUN"].astype(str).isin(rmsp_codes_str)].copy()

In [255]:
zone_table = jobs_by_zone.merge(households_by_zone, on="zona_num", how="outer")
zone_table["job_housing_ratio"] = zone_table["jobs_allocated"] / zone_table["households_allocated"]